# Overview of the Eval Decoders

The BioJEPA-AC model is an encoder meaning its output is an embedding representation, not a specific molecular representation or property. To see the details you can review the [AC Predictor](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_ac_predictor.ipynb).  To complete benchmarks against other models, we need to add a decoder head on top.  

We do need to be careful here though. Our goal with BioJEPA is to build a great generalizable foundation model. Because of this, we need to make sure our head isn't so smart it masks issues in it.  Because of this you'll see our heads are shallow. 

We have 2 heads for our v0.6 evals:
1. Linear expression decoder
2. Linear classifier

![Eval Decoder Overview](../resources/v0_7/eval_decoder_overview.png)


**Linear Expression Decoder**
For Perturb-seq, a common eval is predicting expression values or changes in expression. For our model to do this, we'll have to project from our latent space down to a single value per gene.  Since our cell state latent representation is already $[B, \text{n\_genes}, \text{embed\_dim}]$, this decoder will use a single linear layer to pull the representation down to a single value per gene.  

**Linear Classifier**
BioJEPA has a lot of information in its representation of cells in the latent space. As part of our evals we review how we can extract cell batch information, cell types, if a cell is perturbed, perturbation modes, perturbation pathways. To evaluate input properties from the latent representation, we use a classifier head on top of both the model, and the submodules. To do classifiers on latent representations, we use a linear layer to project from the embedding dimension to the classifier dimensions and mean pooling to get a single value per class.  

This notebook will walk through each decoder we use so that the reader can build an intuition for what each head is doing to the data. To this end, you'll see that we set the layer initializations and numbers to ones where you can hand calculate if you need to follow a layer better.

In [1]:
import torch
import numpy as np
import torch.nn as nn

In [2]:
SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
def mock_data(dim_1, dim_2, low=1, high=9):
    return torch.from_numpy(np.round(np.random.uniform(low, high, size=(dim_1, dim_2)), 0)).float()*0.1

## Linear Expression Decoder

We'll start with the expression decoder.  As mentioned, the goal of this decoder is to generate per-gene expression counts.  While the ACPredictor does output both a mean and logvar, in practice we'll just use the mean for our expression prediction and reserve the logvar for error analysis. 

We'll start by generating the input.  We'll use the same dimensions we used in other explainer notebooks but regenerate the data. 

### Data Prep

We'll start with a simple data prep. The linear expression decoder takes just cell state latents in.   We'll have a batch of 2 to show how two different cells are processed.

In [4]:
batch = 2
num_genes = 8
embed_dim = 6       

In [5]:
cell_latent = mock_data(batch * num_genes, embed_dim).reshape(batch, num_genes, embed_dim)
cell_latent.shape, cell_latent

(torch.Size([2, 8, 6]),
 tensor([[[0.3000, 0.2000, 0.3000, 0.5000, 0.4000, 0.5000],
          [0.3000, 0.9000, 0.7000, 0.2000, 0.4000, 0.6000],
          [0.2000, 0.9000, 0.5000, 0.7000, 0.7000, 0.4000],
          [0.4000, 0.6000, 0.7000, 0.3000, 0.3000, 0.6000],
          [0.5000, 0.2000, 0.4000, 0.3000, 0.5000, 0.8000],
          [0.2000, 0.9000, 0.4000, 0.5000, 0.5000, 0.1000],
          [0.7000, 0.9000, 0.2000, 0.2000, 0.4000, 0.7000],
          [0.1000, 0.4000, 0.1000, 0.7000, 0.4000, 0.7000]],
 
         [[0.5000, 0.9000, 0.8000, 0.6000, 0.6000, 0.9000],
          [0.3000, 0.1000, 0.2000, 0.9000, 0.9000, 0.5000],
          [0.4000, 0.7000, 0.1000, 0.7000, 0.6000, 0.3000],
          [0.6000, 0.8000, 0.2000, 0.8000, 0.4000, 0.7000],
          [0.3000, 0.5000, 0.2000, 0.5000, 0.5000, 0.4000],
          [0.7000, 0.5000, 0.7000, 0.7000, 0.4000, 0.8000],
          [0.7000, 0.8000, 0.4000, 0.5000, 0.8000, 0.6000],
          [0.5000, 0.3000, 0.9000, 0.5000, 0.7000, 0.3000]]]))

### Forward Pass

The goal of this model is to project the cell_latent down to per gene expression, without covering up missing information in the latent. Since the genes in gene expression are our tokens, our task is simple: we just collapse the embedding dimension down to 1 to get a single value per gene. This value becomes our expression representation. 

#### Linear Expression Layer

The linear layer does the projection from the embedding dimensions to 1 so that we have a single value per token (aka gene).  To show how this aggregates across embedding dimensions, we'll do incremental weights so you'll see the impact of the collapsing. 

In [6]:
lin_exp = nn.Linear(embed_dim, 1)
pattern = torch.arange(embed_dim).unsqueeze(0)*1.0
lin_exp.weight = nn.Parameter(pattern)
nn.init.zeros_(lin_exp.bias)
lin_exp.weight, lin_exp.bias

(Parameter containing:
 tensor([[0., 1., 2., 3., 4., 5.]], requires_grad=True),
 Parameter containing:
 tensor([0.], requires_grad=True))

In [7]:
gene_preds = lin_exp(cell_latent) 
gene_preds.shape, gene_preds

(torch.Size([2, 8, 1]),
 tensor([[[ 6.4000],
          [ 7.5000],
          [ 8.8000],
          [ 7.1000],
          [ 7.9000],
          [ 5.7000],
          [ 7.0000],
          [ 7.8000]],
 
         [[11.2000],
          [ 9.3000],
          [ 6.9000],
          [ 8.7000],
          [ 6.4000],
          [ 9.6000],
          [ 9.3000],
          [ 7.9000]]], grad_fn=<ViewBackward0>))

**Expression counts per gene**

We now will remove the embedding dimension. This will give us the output expression per cell. 

In [8]:
gene_preds = gene_preds.squeeze(-1)
gene_preds.shape, gene_preds

(torch.Size([2, 8]),
 tensor([[ 6.4000,  7.5000,  8.8000,  7.1000,  7.9000,  5.7000,  7.0000,  7.8000],
         [11.2000,  9.3000,  6.9000,  8.7000,  6.4000,  9.6000,  9.3000,  7.9000]],
        grad_fn=<SqueezeBackward1>))

#### Expression Counts


With just that single layer and reshape, we've now calculated the predicted expression values per gene. As a reminder, these are not absolute expression counts, but expression counts aligned with our data prep, meaning they represent count and log normalized expression. 

## Linear Classifier

We'll now show how our linear classifier works. As mentioned, our linear classifier for evals is built to be used across a number of input metadata such as cell batch information, cell types, if a cell is perturbed, perturbation modes, perturbation pathways. The goal of our classifier is to predict a probability that each example in the input fits into each class. As we dive into each eval, you'll see how the input and classes change. 

We'll start by generating the input that represents a cell state latent again and generate a number of classes. We'll use the same dimensions we used in other explainer notebooks but regenerate the data.

### Data Prep

We'll start with a simple data prep. The classifier we'll build will take just cell state latents in. We'll need to preconfigure the number of classes. We'll also use a batch of 2 to show how two different cells are processed.

In [9]:
batch = 2
num_genes = 8
embed_dim = 6
num_classes = 3

In [10]:
cell_latent = mock_data(batch * num_genes, embed_dim).reshape(batch, num_genes, embed_dim)
cell_latent.shape, cell_latent

(torch.Size([2, 8, 6]),
 tensor([[[0.4000, 0.4000, 0.5000, 0.2000, 0.6000, 0.1000],
          [0.3000, 0.7000, 0.8000, 0.6000, 0.4000, 0.8000],
          [0.9000, 0.3000, 0.4000, 0.7000, 0.4000, 0.1000],
          [0.4000, 0.5000, 0.1000, 0.8000, 0.7000, 0.7000],
          [0.2000, 0.1000, 0.3000, 0.2000, 0.3000, 0.6000],
          [0.5000, 0.8000, 0.6000, 0.2000, 0.2000, 0.4000],
          [0.4000, 0.2000, 0.6000, 0.1000, 0.7000, 0.6000],
          [0.1000, 0.7000, 0.4000, 0.7000, 0.4000, 0.6000]],
 
         [[0.2000, 0.8000, 0.3000, 0.7000, 0.6000, 0.8000],
          [0.5000, 0.1000, 0.2000, 0.9000, 0.5000, 0.2000],
          [0.6000, 0.6000, 0.5000, 0.8000, 0.8000, 0.3000],
          [0.7000, 0.5000, 0.3000, 0.1000, 0.3000, 0.6000],
          [0.3000, 0.5000, 0.4000, 0.5000, 0.6000, 0.9000],
          [0.1000, 0.5000, 0.6000, 0.2000, 0.7000, 0.4000],
          [0.5000, 0.6000, 0.8000, 0.7000, 0.6000, 0.1000],
          [0.1000, 0.5000, 0.7000, 0.7000, 0.3000, 0.3000]]]))

### Forward Pass

The goal of this model is to project the input latent (cell_latent) down to per-class probabilities per input example, without covering up missing information in the latent. 

Our input cell latent has $[\text{batch},\text{n\_genes},\text{embed\_dim}]$ so you'll see that a single linear layer will give us a per-class projection for each "middle" dimension, in this input case the genes.  To get this down to a single value without adding more intelligence to cover up deficits, we do simple mean pooling to then pull the output to $[\text{batch},\text{class}]$

#### Linear Classifier Layer

The linear layer does the projection from the embedding dimensions to our number of classes so that we have a per-class value for each token (aka gene). To show how this aggregates across embedding dimensions, we'll do incremental weights so you'll see the impact of the collapsing. 

In [11]:
class_pred = nn.Linear(embed_dim, num_classes)
vs, d = embed_dim, num_classes
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.1*cols)  
class_pred.weight = nn.Parameter(pattern)
nn.init.zeros_(class_pred.bias)
class_pred.weight, class_pred.bias

(Parameter containing:
 tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
         [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0.], requires_grad=True))

In [12]:
lin_class = class_pred(cell_latent)
lin_class.shape, lin_class

(torch.Size([2, 8, 3]),
 tensor([[[0.2200, 0.4400, 0.6600],
          [0.3600, 0.7200, 1.0800],
          [0.2800, 0.5600, 0.8400],
          [0.3200, 0.6400, 0.9600],
          [0.1700, 0.3400, 0.5100],
          [0.2700, 0.5400, 0.8100],
          [0.2600, 0.5200, 0.7800],
          [0.2900, 0.5800, 0.8700]],
 
         [[0.3400, 0.6800, 1.0200],
          [0.2400, 0.4800, 0.7200],
          [0.3600, 0.7200, 1.0800],
          [0.2500, 0.5000, 0.7500],
          [0.3200, 0.6400, 0.9600],
          [0.2500, 0.5000, 0.7500],
          [0.3300, 0.6600, 0.9900],
          [0.2600, 0.5200, 0.7800]]], grad_fn=<ViewBackward0>))

#### (Optional) Mean pooling

Now that we have our class probabilities per gene, we're ready to collapse down to a single value per example. We do this by calculating the mean of each class to get a single numeric value.  This helps smooth out high and low outliers for the class. 

*Note that this step can be skipped if there aren't 3 dimensions, like in the case of classifiers that we run on top of the action composer output.*

In [13]:
lin_class = lin_class.mean(dim=1)
lin_class.shape, lin_class

(torch.Size([2, 3]),
 tensor([[0.2713, 0.5425, 0.8138],
         [0.2938, 0.5875, 0.8813]], grad_fn=<MeanBackward1>))

#### Class probabilities

With just the single linear layer and, when needed, mean pooling, we've gone from our input into a per example classifier.  You may look at the numbers and, correctly, point out they don't add up to 1 so they're not real probabilities. That's fine since each eval will use this differently.  Some of our evals will just pick the highest value if we need a class, others will convert to real probabilities to calculate AUROC.  By keeping our classifier generic and outputting raw values, we create a flexible class we can use across our different evals. 